# 寒武纪 (688256.SH) 近一年行情分析

**数据来源：Tushare Pro**  
**区间：2025-07-01 至 2026-07-01**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import json

# 中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# 中国惯例：涨红跌绿
UP_COLOR = '#ff4d4f'
DOWN_COLOR = '#52c41a'

## 1. 数据加载与概览

In [ ]:
# 加载数据
df = pd.read_csv('cambricon_daily_data.csv')
df['trade_date'] = pd.to_datetime(df['trade_date'], format='%Y%m%d')
df = df.sort_values('trade_date').reset_index(drop=True)

print(f"数据条数: {len(df)}")
print(f"起始日期: {df['trade_date'].min().date()}")
print(f"结束日期: {df['trade_date'].max().date()}")
df.head()

In [ ]:
# 基础统计
stats = df[['open','high','low','close','vol','amount']].describe()
stats.round(2)

## 2. 价格走势与波动分析

In [ ]:
# 计算收益率
df['return'] = df['close'].pct_change() * 100
df['volatility_20'] = df['return'].rolling(20).std()  # 20日波动率

# 区间涨跌
start_price = df['close'].iloc[0]
end_price = df['close'].iloc[-1]
total_return = (end_price - start_price) / start_price * 100
max_price = df['high'].max()
min_price = df['low'].min()
max_dd = (df['close'].cummax() - df['close']) / df['close'].cummax() * 100

print(f"区间涨跌幅: {total_return:+.2f}%")
print(f"区间最高价: ¥{max_price:.2f} ({df.loc[df['high'].idxmax(), 'trade_date'].date()})")
print(f"区间最低价: ¥{min_price:.2f} ({df.loc[df['low'].idxmin(), 'trade_date'].date()})")
print(f"最大回撤: {max_dd.max():.2f}% ({df.loc[max_dd.idxmax(), 'trade_date'].date()})")
print(f"日均收益率: {df['return'].mean():.4f}%")
print(f"年化波动率: {df['return'].std() * np.sqrt(252):.2f}%")

## 3. K线图 + 成交量

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 10), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
fig.suptitle('寒武纪 (688256.SH) 近一年K线图', fontsize=18, fontweight='bold')

# K线图
width = 0.6
for i, row in df.iterrows():
    color = UP_COLOR if row['close'] >= row['open'] else DOWN_COLOR
    # 影线
    ax1.plot([i, i], [row['low'], row['high']], color=color, linewidth=0.8)
    # 实体
    body_bottom = min(row['open'], row['close'])
    body_height = abs(row['close'] - row['open'])
    ax1.add_patch(Rectangle((i - width/2, body_bottom), width, max(body_height, 0.1),
                            facecolor=color if row['close'] >= row['open'] else 'white',
                            edgecolor=color, linewidth=0.5))

# 均线
for n, clr, lbl in [(5, '#e6a23c', 'MA5'), (10, '#409eff', 'MA10'), (20, '#9b59b6', 'MA20'), (30, '#2ecc71', 'MA30')]:
    ma = df['close'].rolling(n).mean()
    ax1.plot(ma.values, color=clr, linewidth=1.2, label=lbl)

ax1.legend(loc='upper left', fontsize=9)
ax1.set_ylabel('价格 (¥)', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-1, len(df))

# 成交量
colors = [UP_COLOR if row['close'] >= row['open'] else DOWN_COLOR for _, row in df.iterrows()]
ax2.bar(range(len(df)), df['vol'] / 10000, color=colors, width=0.8, alpha=0.85)
ax2.set_ylabel('成交量 (万手)', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-1, len(df))

# X轴标签（只显示月份）
month_ticks = []
month_labels = []
for i, row in df.iterrows():
    label = row['trade_date'].strftime('%Y-%m')
    if not month_labels or month_labels[-1] != label:
        month_ticks.append(i)
        month_labels.append(label)

ax2.set_xticks(month_ticks)
ax2.set_xticklabels([m[2:] for m in month_labels], rotation=45, fontsize=9)
ax2.set_xlabel('日期', fontsize=12)

plt.tight_layout()
plt.show()

## 4. 收益率分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 日收益率时序
ax = axes[0]
pos = df['return'] >= 0
ax.bar(df.index[pos], df['return'][pos], color=UP_COLOR, width=0.8, alpha=0.8, label='上涨')
ax.bar(df.index[~pos], df['return'][~pos], color=DOWN_COLOR, width=0.8, alpha=0.8, label='下跌')
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_title('日收益率时序图', fontsize=14, fontweight='bold')
ax.set_ylabel('收益率 (%)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 收益率直方图
ax = axes[1]
ax.hist(df['return'].dropna(), bins=30, color='#0f3460', edgecolor='white', alpha=0.8)
ax.axvline(0, color='red', linestyle='--', linewidth=1, label='零线')
ax.axvline(df['return'].mean(), color='orange', linestyle='--', linewidth=1, label=f'均值={df["return"].mean():.2f}%')
ax.set_title('日收益率分布', fontsize=14, fontweight='bold')
ax.set_xlabel('收益率 (%)')
ax.set_ylabel('频次')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

up_days = (df['return'] > 0).sum()
down_days = (df['return'] < 0).sum()
print(f"上涨天数: {up_days} ({up_days/len(df)*100:.1f}%)")
print(f"下跌天数: {down_days} ({down_days/len(df)*100:.1f}%)")

plt.tight_layout()
plt.show()

## 5. 滚动波动率与回撤

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 8))

# 价格 + 20日滚动波动率
ax1.fill_between(df.index, df['close'], alpha=0.1, color='#0f3460')
ax1.plot(df['close'], color='#0f3460', linewidth=1.5, label='收盘价')
ax1_twin = ax1.twinx()
ax1_twin.plot(df['volatility_20'], color='orange', linewidth=1, label='20日波动率')
ax1_twin.fill_between(df.index, df['volatility_20'], alpha=0.1, color='orange')
ax1.set_title('价格走势与20日滚动波动率', fontsize=14, fontweight='bold')
ax1.set_ylabel('价格 (¥)')
ax1_twin.set_ylabel('波动率 (%)')
ax1.grid(True, alpha=0.3)

# 回撤曲线
ax2.fill_between(df.index, max_dd, color=DOWN_COLOR, alpha=0.3)
ax2.plot(df.index, max_dd, color=DOWN_COLOR, linewidth=1.2)
ax2.set_title('最大回撤', fontsize=14, fontweight='bold')
ax2.set_ylabel('回撤 (%)')
ax2.invert_yaxis()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 月度表现

In [ ]:
# 按月份聚合
df_monthly = df.set_index('trade_date').resample('ME').agg({
    'open': 'first',
    'close': 'last',
    'high': 'max',
    'low': 'min',
    'vol': 'sum'
})
df_monthly['return'] = df_monthly['close'].pct_change() * 100
df_monthly['month_label'] = df_monthly.index.strftime('%Y-%m')

fig, ax = plt.subplots(figsize=(16, 5))
colors = [UP_COLOR if v >= 0 else DOWN_COLOR for v in df_monthly['return'].fillna(0)]
bars = ax.bar(range(len(df_monthly)), df_monthly['return'].fillna(0), color=colors, edgecolor='white', width=0.65)
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_xticks(range(len(df_monthly)))
ax.set_xticklabels(df_monthly['month_label'], rotation=45, fontsize=10)
ax.set_title('月度涨跌幅', fontsize=14, fontweight='bold')
ax.set_ylabel('涨跌幅 (%)')
ax.grid(True, alpha=0.3, axis='y')

for i, (v, c) in enumerate(zip(df_monthly['return'].fillna(0), colors)):
    ax.text(i, v + (1 if v >= 0 else -1), f'{v:+.1f}%', ha='center', va='bottom' if v >= 0 else 'top', fontsize=8, color=c)

plt.tight_layout()
plt.show()

df_monthly[['month_label','open','close','high','low','return']].round(2)

---
**免责声明：本分析仅供学习参考，不构成任何投资建议。入市有风险，投资需谨慎。**